In [2]:
import numpy as np
import pandas as pd

# Load Data

In [3]:
train = pd.read_csv("data/train.csv")
transactions = pd.read_csv("data/transactions.csv")
stores = pd.read_csv("data/stores.csv")
items = pd.read_csv("data/items.csv")
oil = pd.read_csv("data/oil.csv")
holidays = pd.read_csv("data/holidays_events.csv")

print("Train:", train.shape)
print("Transactions:", transactions.shape)
print("Stores:", stores.shape)
print("Items:", items.shape)
print("Oil:", oil.shape)
print("Holidays:", holidays.shape)

C:\Users\HP\AppData\Local\Temp\ipykernel_7832\202290072.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("data/train.csv")


Train: (125497040, 6)
Transactions: (83488, 3)
Stores: (54, 5)
Items: (4100, 4)
Oil: (1218, 2)
Holidays: (350, 6)


# Filtering Store

In [4]:
selected_stores = [25, 26, 27, 28, 29]

train_filtered = train[
    train["store_nbr"].isin(selected_stores)
].copy()

print("Filtered train:", train_filtered.shape)

Filtered train: (10409591, 6)


# Cleaning Data #1

In [5]:
train_filtered["date"] = pd.to_datetime(train_filtered["date"])
transactions["date"] = pd.to_datetime(transactions["date"])
stores = stores.copy()
items = items.copy()
oil["date"] = pd.to_datetime(oil["date"])
holidays["date"] = pd.to_datetime(holidays["date"])

In [6]:
print("Duplicate ID:",
      train_filtered["id"].duplicated().sum())

print("Negative sales:",
      (train_filtered["unit_sales"] < 0).sum())

Duplicate ID: 0
Negative sales: 586


# Merge Store's Information and Train

In [7]:
train_filtered = train_filtered.merge(
    stores[
        [
            "store_nbr",
            "city",
            "state",
            "type",
            "cluster"
        ]
    ],
    on="store_nbr",
    how="left"
)

# Merge Item's Information and Train

In [8]:
train_filtered = train_filtered.merge(
    items[
        [
            "item_nbr",
            "family",
            "class",
            "perishable"
        ]
    ],
    on="item_nbr",
    how="left"
)

In [9]:
print(train_filtered.columns.tolist())

['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'family', 'class', 'perishable']


# Merge Oil Price and Train

In [10]:
oil = oil.sort_values("date")

oil["dcoilwtico"] = oil["dcoilwtico"].interpolate()
oil["dcoilwtico"] = oil["dcoilwtico"].ffill()
oil["dcoilwtico"] = oil["dcoilwtico"].bfill()

train_filtered = train_filtered.merge(
    oil[["date", "dcoilwtico"]],
    on="date",
    how="left"
)

train_filtered = train_filtered.rename(
    columns={"dcoilwtico": "oil_price"}
)

# Holidays Features

In [11]:
holiday_dates = holidays[
    holidays["type"].isin(
        ["Holiday", "Transfer", "Additional"]
    )
]["date"].unique()

# Merge holidays and train
train_filtered["is_holiday"] = (
    train_filtered["date"]
    .isin(holiday_dates)
    .astype("int8")
)

# Merge Transactions and Train

In [12]:
transactions = transactions[
    ["date", "store_nbr", "transactions"]
].copy()

transactions["transactions"] = (
    transactions["transactions"]
    .fillna(0)
    .astype("float32")
)

train_filtered = train_filtered.merge(
    transactions,
    on=["date", "store_nbr"],
    how="left"
)

train_filtered["transactions"] = (
    train_filtered["transactions"]
    .fillna(0)
    .astype("float32")
)

# Calendar Features

In [13]:
train_filtered["year"] = (
    train_filtered["date"].dt.year
)

train_filtered["month"] = (
    train_filtered["date"].dt.month
)

train_filtered["day"] = (
    train_filtered["date"].dt.day
)

train_filtered["day_of_week"] = (
    train_filtered["date"].dt.dayofweek
)

train_filtered["week_of_year"] = (
    train_filtered["date"].dt.isocalendar().week
    .astype("int8")
)

train_filtered["is_weekend"] = (
    train_filtered["day_of_week"] >= 5
).astype("int8")

# Data Sorting before Lag

In [14]:
train_sorted = train_filtered.sort_values(
    ["store_nbr", "item_nbr", "date"]
).copy()

# Targetting

In [15]:
# prediksi jumlah produk yg akan terjual

train_sorted["sales_target"] = (
    train_sorted["unit_sales"]
    .clip(lower=0)
    .astype("float32")
)

# Sales Lag Feature

In [16]:
group_cols = ["store_nbr", "item_nbr"]

# lag 1
train_sorted["lag_1"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .shift(1)
)

# lag 7 hari
train_sorted["lag_7"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .shift(7)
)

# lag 14 hari
train_sorted["lag_14"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .shift(14)
)

# lag 21 hari
train_sorted["lag_21"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .shift(21)
)

# lag 28 hari
train_sorted["lag_28"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .shift(28)
)

# lag 30 hari
train_sorted["lag_30"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .shift(30)
)

# Rolling Features

In [17]:
# Rolling mean 7 hari
train_sorted["rolling_mean_7"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

# 30 hari
train_sorted["rolling_mean_30"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .transform(
        lambda x:
        x.shift(1).rolling(30).mean()
    )
)

# median
train_sorted["rolling_median_7"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).median()
    )
)

# standar deviasi
train_sorted["rolling_std_7"] = (
    train_sorted
    .groupby(group_cols)["sales_target"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).std()
    )
)

# Transactions Features

In [18]:
# dataframe khusus store-day
trans_features = transactions[
    ["date", "store_nbr", "transactions"]
].copy()

trans_features = trans_features.sort_values(
    ["store_nbr", "date"]
)

In [19]:
# trans lag 1 hari
trans_features["transactions_lag_1"] = (
    trans_features
    .groupby("store_nbr")["transactions"]
    .shift(1)
)

# trans lag 7 hari
trans_features["transactions_lag_7"] = (
    trans_features
    .groupby("store_nbr")["transactions"]
    .shift(7)
)

# trans rolling 7
trans_features["transactions_rolling_7"] = (
    trans_features
    .groupby("store_nbr")["transactions"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

# Merge Transactions and Sales

In [20]:
train_sorted = train_sorted.merge(
    trans_features[
        [
            "date",
            "store_nbr",
            "transactions_lag_1",
            "transactions_lag_7",
            "transactions_rolling_7"
        ]
    ],
    on=["date", "store_nbr"],
    how="left"
)

# Promotion Features

In [21]:
# history promosi
train_sorted["promotion_lag_1"] = (
    train_sorted
    .groupby(group_cols)["onpromotion"]
    .shift(1)
)

# jml promosi 7 hari terakhir
train_sorted["promotion_rolling_7"] = (
    train_sorted
    .groupby(group_cols)["onpromotion"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).sum()
    )
)

# Check Missing Values

In [22]:
lag_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_21",
    "lag_28",
    "lag_30",
    "rolling_mean_7",
    "rolling_mean_30",
    "rolling_median_7",
    "rolling_std_7",
    "transactions_lag_1",
    "transactions_lag_7",
    "transactions_rolling_7",
    "promotion_lag_1",
    "promotion_rolling_7"
]

print(
    train_sorted[lag_features]
    .isna()
    .sum()
)

lag_1                       16791
lag_7                      116478
lag_14                     231381
lag_21                     344895
lag_28                     457191
lag_30                     489086
rolling_mean_7             116478
rolling_mean_30            489086
rolling_median_7           116478
rolling_std_7              116478
transactions_lag_1          21969
transactions_lag_7          49002
transactions_rolling_7      49002
promotion_lag_1           1729794
promotion_rolling_7       1829178
dtype: int64


In [23]:
train_sorted = train_sorted.dropna(
    subset=lag_features
).copy()

# Log Transformed

In [24]:
train_sorted["target_log"] = np.log1p(
    train_sorted["sales_target"]
)

# Tentukan Features

In [25]:
features = [
    # Store / Item
    "store_nbr",
    "item_nbr",

    # Calendar
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "is_weekend",

    # Business
    "onpromotion",
    "is_holiday",
    "oil_price",

    # Store / Item information
    "perishable",
    "cluster",
    "class",

    # Sales history
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_21",
    "lag_28",
    "lag_30",

    # Rolling statistics
    "rolling_mean_7",
    "rolling_mean_30",
    "rolling_median_7",
    "rolling_std_7",

    # Transactions
    "transactions_lag_1",
    "transactions_lag_7",
    "transactions_rolling_7",

    # Promotion history
    "promotion_lag_1",
    "promotion_rolling_7"
]

target = "target_log"

missing_features = [
    col for col in features
    if col not in train_sorted.columns
]

print("Missing features:", missing_features)

Missing features: []


# Change OnPromotion from object to numeric

In [26]:
# Ubah fitur promotion menjadi numerik
train_sorted["onpromotion"] = pd.to_numeric(
    train_sorted["onpromotion"],
    errors="coerce"
)

train_sorted["promotion_lag_1"] = pd.to_numeric(
    train_sorted["promotion_lag_1"],
    errors="coerce"
)

train_sorted["promotion_rolling_7"] = pd.to_numeric(
    train_sorted["promotion_rolling_7"],
    errors="coerce"
)

In [27]:
train_sorted["onpromotion"] = (
    train_sorted["onpromotion"]
    .fillna(0)
    .astype("int8")
)

train_sorted["promotion_lag_1"] = (
    train_sorted["promotion_lag_1"]
    .fillna(0)
    .astype("int8")
)

train_sorted["promotion_rolling_7"] = (
    train_sorted["promotion_rolling_7"]
    .fillna(0)
    .astype("float32")
)

In [28]:
print(
    train_sorted[features].dtypes
)

store_nbr                   int64
item_nbr                    int64
year                        int32
month                       int32
day                         int32
day_of_week                 int32
week_of_year                 int8
is_weekend                   int8
onpromotion                  int8
is_holiday                   int8
oil_price                 float64
perishable                  int64
cluster                     int64
class                       int64
lag_1                     float32
lag_7                     float32
lag_14                    float32
lag_21                    float32
lag_28                    float32
lag_30                    float32
rolling_mean_7            float64
rolling_mean_30           float64
rolling_median_7          float64
rolling_std_7             float64
transactions_lag_1        float32
transactions_lag_7        float32
transactions_rolling_7    float64
promotion_lag_1              int8
promotion_rolling_7       float32
dtype: object


In [29]:
print(
    train_sorted[features]
    .select_dtypes(include="object")
    .columns
)

Index([], dtype='object')


In [30]:
print(
    train_sorted[features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

oil_price       2588389
item_nbr              0
year                  0
month                 0
store_nbr             0
day                   0
day_of_week           0
is_weekend            0
week_of_year          0
onpromotion           0
is_holiday            0
perishable            0
cluster               0
class                 0
lag_1                 0
lag_7                 0
lag_14                0
lag_21                0
lag_28                0
lag_30                0
dtype: int64


In [31]:
train_sorted = train_sorted.dropna(
    subset=features
).copy()

# Split

In [32]:
split_date = train_sorted["date"].quantile(0.8)

train_data = train_sorted[
    train_sorted["date"] <= split_date
].copy()

test_data = train_sorted[
    train_sorted["date"] > split_date
].copy()

# XGBoost Modeling

In [33]:
X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test_log = test_data[target]

# Target asli untuk evaluasi
y_test = test_data["sales_target"]

In [34]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    objective="reg:squarederror",

    n_estimators=1000,
    learning_rate=0.03,

    max_depth=8,
    min_child_weight=5,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1
)

# Prediction

In [35]:
xgb.fit(
    X_train,
    y_train
)

y_pred_log = xgb.predict(X_test)

# Prediksi sales tidak boleh negatif
y_pred = np.expm1(y_pred_log)

y_pred = np.maximum(
    y_pred,
    0
)

# XGBoost Model Eval

In [36]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

wape = (
    np.sum(np.abs(y_test - y_pred))
    /
    np.sum(np.abs(y_test))
) * 100

r2 = r2_score(
    y_test,
    y_pred
)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"WAPE : {wape:.2f}%")
print(f"R²   : {r2:.4f}")

MAE  : 2.6808
RMSE : 10.5653
WAPE : 45.33%
R²   : 0.4315


# Save Model

In [37]:
import joblib

joblib.dump(xgb, "xgboost_sales_forecasting.pkl")

['xgboost_sales_forecasting.pkl']

In [38]:
joblib.dump(
    X_train.columns.tolist(),
    "feature_columns.pkl"
)

['feature_columns.pkl']

In [39]:
metadata = {
    "target": "sales_target",
    "target_transform": "log1p",
    "prediction_inverse": "expm1",
    "lookback": None,
    "model_type": "XGBoost",
    "features": X_train.columns.tolist()
}

joblib.dump(
    metadata,
    "model_metadata.pkl"
)

['model_metadata.pkl']

In [43]:
streamlit_data = train_sorted[
    [
        "date",
        "store_nbr",
        "item_nbr",
        "sales_target",
        "onpromotion",
        "transactions",
        "family",
        "city",
        "state"
    ]
].copy()

streamlit_data.to_csv(
    "train_processed.csv",
    index=False
)

In [44]:
import os

print(os.path.getsize("train_processed.csv") / (1024**3), "GB")

0.331811991520226 GB
